# B03 — Type 1 Logic Eval (via `/predict` API)

Sends each inference instance to the running EXACT API `/predict` endpoint using
the **unified input schema**, then scores predictions against gold answers.

- **Requests:** `Logic_Based_Educational_Queries_inference.json` (ground truth removed)
- **Gold:** `Logic_Based_Educational_Queries.json`, joined by `group_id` → list
  index and `question_index`.
- **Label note:** gold uses `"Unknown"`; the competition label is `"Uncertain"` —
  scored as equal here.
- **Options note:** MCQ vs YNU is detected from the *question text* (presence of
  `A.`/`B.`/`C.`/`D.`), because `question_index` does not map cleanly to type.


In [ ]:
import json, re, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

# --- endpoint -------------------------------------------------------------
API_BASE    = "http://127.0.0.1:8080"        # local; or "https://api.iamphuckhang.dev"
PREDICT_URL = f"{API_BASE}/predict"

# --- run size -------------------------------------------------------------
N_SAMPLES   = 20        # how many instances to eval (None = all 808)
CONCURRENCY = 8         # parallel in-flight requests
TIMEOUT     = 120.0     # per-request seconds

# --- locate dataset dir (walk up to project root) -------------------------
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", PREDICT_URL)

In [ ]:
inf  = json.load(open(DATA / "Logic_Based_Educational_Queries_inference.json"))["instances"]
orig = json.load(open(DATA / "Logic_Based_Educational_Queries.json"))
print(f"{len(inf)} inference instances, {len(orig)} original groups")


def gold_for(inst: dict) -> str:
    g = int(inst["group_id"].split("_")[1])
    return orig[g]["answers"][inst["question_index"]]


def detect_options(question: str) -> list[str]:
    # MCQ if the question lists choices like 'A.' 'B.' ...; else Yes/No/Uncertain.
    if re.search(r"(^|\n)\s*[A-D][.)]", question):
        return ["A", "B", "C", "D"]
    return ["Yes", "No", "Uncertain"]


def build_request(inst: dict) -> dict:
    # Unified input schema. The API accepts query_id/query as aliases for id/question.
    return {
        "query_id": inst["id"],
        "type": "type1",
        "query": inst["question"],
        "premises": inst["premises-NL"],
        "options": detect_options(inst["question"]),
    }


def normalize(ans) -> str:
    a = str(ans or "").strip().upper()
    if a in {"UNKNOWN", "UNCERTAIN"}:
        return "UNCERTAIN"
    m = re.match(r"^([A-D])\b", a)       # leading MCQ letter (maybe '.'/text after)
    if m:
        return m.group(1)
    if a.startswith("YES"):
        return "YES"
    if a.startswith("NO"):
        return "NO"
    return a


# sanity check on one item
ex = inf[0]
print(json.dumps(build_request(ex), ensure_ascii=False, indent=2)[:400])
print("gold:", gold_for(ex), "-> norm:", normalize(gold_for(ex)))

In [ ]:
async def call(client, sem, inst):
    payload = build_request(inst)
    gold = gold_for(inst)
    async with sem:
        t0 = time.perf_counter()
        err, data, pred = None, {}, ""
        try:
            r = await client.post(PREDICT_URL, json=payload, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
            pred = data.get("answer", "")
        except Exception as e:
            err = repr(e)
        dt = time.perf_counter() - t0
    return {
        "id": inst["id"],
        "pred": pred,
        "gold": gold,
        "ok": normalize(pred) == normalize(gold),
        "latency": dt,
        "error": err,
        "raw": data,
    }


async def run_eval(instances):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(inst):
            nonlocal done
            res = await call(client, sem, inst)
            done += 1
            if done % 10 == 0 or done == len(instances):
                print(f"  {done}/{len(instances)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(i) for i in instances))

In [ ]:
subset = inf[:N_SAMPLES] if N_SAMPLES else inf
print(f"Evaluating {len(subset)} instances at concurrency {CONCURRENCY}...")
t0 = time.perf_counter()
results = await run_eval(subset)          # Jupyter supports top-level await
wall = time.perf_counter() - t0

correct = sum(r["ok"] for r in results)
errors  = [r for r in results if r["error"]]
print(f"\nAccuracy: {correct}/{len(results)} = {correct / len(results):.1%}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")

In [ ]:
# Mismatches and errors
for r in results:
    if not r["ok"]:
        tag = "ERR  " if r["error"] else "WRONG"
        print(f"[{tag}] {r['id']}  pred={r['pred']!r:<14} gold={r['gold']!r:<10} {r['error'] or ''}")

In [ ]:
# Breakdown by question type + latency
by = {}
for inst, r in zip(subset, results):
    t = "MCQ" if detect_options(inst["question"]) == ["A", "B", "C", "D"] else "YNU"
    by.setdefault(t, []).append(r["ok"])
for t, v in sorted(by.items()):
    print(f"{t}: {sum(v)}/{len(v)} = {sum(v) / len(v):.1%}")

lat = [r["latency"] for r in results]
print(f"latency mean={statistics.mean(lat):.1f}s  p50={statistics.median(lat):.1f}s  max={max(lat):.1f}s")
print("pred distribution:", Counter(normalize(r["pred"]) for r in results))

## Notes
- Set `N_SAMPLES = None` to run the full 808 (slow: roughly `N / CONCURRENCY × mean_latency`).
- Point `API_BASE` at the public domain to test through the cloudflared tunnel,
  but local `127.0.0.1:8080` avoids Cloudflare's request timeout on slow items.
- `normalize()` extracts the leading MCQ letter and treats `Unknown == Uncertain`.
  A full-text answer that omits the letter counts as wrong — refine `normalize()`
  if the model returns prose instead of a label.
- To inspect a single failure: `next(r for r in results if not r['ok'])['raw']`.
